# 04 Models
Logistic regression first, then gradient boosting. Train on 2022–23 and tune on 2024 only. 2025 stays locked.

We report a model as better only if it beats the baseline on **both** hits per race and Brier score.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluate import baseline_prob, evaluate, fit_grid_baseline, load_features, podium_hits, time_split
from src.features import FEATURES

train, valid = time_split(load_features())
X_train, y_train = train[FEATURES], train["Podium"]
X_valid, y_valid = valid[FEATURES], valid["Podium"]

grid_rate = fit_grid_baseline(train)
valid["BaselineProb"] = baseline_prob(valid, grid_rate)
results = {"baseline (grid top 3)": evaluate(valid, "BaselineProb")}
pd.DataFrame(results)

,baseline (grid top 3)
hits_per_race,2.0000
perfect_races_pct,20.8000
brier,0.0699


## Missing values
Form features are NaN in a driver's or team's first race. Imputation uses medians from **train only** and adds missing-value flags, so the model can tell "no history yet" apart from "average".

In [2]:
def make_logreg(C):
    return make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True),
        StandardScaler(),
        LogisticRegression(C=C, max_iter=2000),
    )

logreg_scores = {}
for C in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]:
    model = make_logreg(C).fit(X_train, y_train)
    valid["Prob"] = model.predict_proba(X_valid)[:, 1]
    logreg_scores[C] = evaluate(valid, "Prob")

pd.DataFrame(logreg_scores).T

,hits_per_race,perfect_races_pct,brier
0.01,1.667,4.2,0.0798
0.03,1.750,4.2,0.0764
0.10,1.917,16.7,0.0741
0.30,1.958,20.8,0.0733
1.00,2.042,29.2,0.0730
3.00,2.042,29.2,0.0729


Pick C by Brier score: it's smoother than hits, which only move in steps of 1/24.

In [3]:
best_C = min(logreg_scores, key=lambda c: logreg_scores[c]["brier"])
logreg = make_logreg(best_C).fit(X_train, y_train)
valid["LogRegProb"] = logreg.predict_proba(X_valid)[:, 1]
results[f"logreg (C={best_C})"] = evaluate(valid, "LogRegProb")

coefs = pd.Series(logreg[-1].coef_[0], index=logreg[:-1].get_feature_names_out()).sort_values()
print(f"best C = {best_C}")
coefs.round(2)

best C = 3.0


GridPosition                        -1.90
DriverAvgGrid                       -0.40
TeamAvgFinish                       -0.35
DriverAvgFinish                     -0.21
GapToPolePct                        -0.14
NoQualiTime                         -0.13
PitLaneStart                        -0.07
missingindicator_DriverPodiumRate   -0.00
missingindicator_DriverAvgGrid      -0.00
missingindicator_DriverAvgFinish    -0.00
DriverPodiumRate                     0.12
missingindicator_TeamPodiumRate      0.13
missingindicator_TeamAvgFinish       0.13
RacesSoFar                           0.13
GridVsTeammate                       0.17
TeamPodiumRate                       0.30
dtype: float64

## Gradient boosting
HistGradientBoosting handles NaN natively, so no imputer is needed. The grid is small on purpose: with only 44 training races, a bigger search would overfit 2024.

In [4]:
gb_scores = {}
for lr in [0.03, 0.1]:
    for depth in [2, 3]:
        for leaf in [20, 50]:
            model = HistGradientBoostingClassifier(
                learning_rate=lr, max_depth=depth, min_samples_leaf=leaf,
                max_iter=200, l2_regularization=1.0, random_state=42,
            ).fit(X_train, y_train)
            valid["Prob"] = model.predict_proba(X_valid)[:, 1]
            gb_scores[(lr, depth, leaf)] = evaluate(valid, "Prob")

pd.DataFrame(gb_scores).T.sort_values("brier")

hits_per_race  perfect_races_pct   brier
0.03 2 20          1.875               16.7  0.0743
       50          1.917               20.8  0.0749
     3 50          1.917               20.8  0.0762
       20          1.917               20.8  0.0765
0.10 2 20          1.875               16.7  0.0766
       50          1.917               16.7  0.0775
     3 20          1.958               20.8  0.0802
       50          1.917               16.7  0.0811

In [5]:
best_gb = min(gb_scores, key=lambda k: gb_scores[k]["brier"])
lr, depth, leaf = best_gb
gb = HistGradientBoostingClassifier(
    learning_rate=lr, max_depth=depth, min_samples_leaf=leaf,
    max_iter=200, l2_regularization=1.0, random_state=42,
).fit(X_train, y_train)
valid["GBProb"] = gb.predict_proba(X_valid)[:, 1]
results[f"gboost {best_gb}"] = evaluate(valid, "GBProb")

## Fixes
Both models above lose to the baseline. Two reasons:
1. **Grid is non-linear.** The podium rate falls sharply from P1 to P3 to P8, but logistic regression treats grid as a straight line. Fix: add `LogGrid` and `InvGrid`.
2. **Form isn't relative to the race.** A team podium rate of 0.6 means something different depending on the field. Fix: rank form features within each race (1 = best).

Both use only pre-race info from the same race, so there's no leakage.

In [6]:
import numpy as np

RANK_COLS = {  # column -> higher is better?
    "TeamPodiumRate": True, "DriverPodiumRate": True,
    "TeamAvgFinish": False, "DriverAvgFinish": False, "GapToPolePct": False,
}

def add_race_features(df):
    df = df.copy()
    df["LogGrid"] = np.log(df["GridPosition"])
    df["InvGrid"] = 1 / df["GridPosition"]
    for col, higher_better in RANK_COLS.items():
        df[f"{col}Rank"] = df.groupby(["Year", "Round"])[col].rank(ascending=not higher_better)
    return df

FEATURES_V2 = FEATURES + ["LogGrid", "InvGrid"] + [f"{c}Rank" for c in RANK_COLS]
train, valid = add_race_features(train), add_race_features(valid)
X_train2, X_valid2 = train[FEATURES_V2], valid[FEATURES_V2]

logreg2_scores = {}
for C in [0.03, 0.1, 0.3, 1.0, 3.0]:
    model = make_logreg(C).fit(X_train2, y_train)
    valid["Prob"] = model.predict_proba(X_valid2)[:, 1]
    logreg2_scores[C] = evaluate(valid, "Prob")
pd.DataFrame(logreg2_scores).T

,hits_per_race,perfect_races_pct,brier
0.03,1.958,25.0,0.0684
0.10,1.917,20.8,0.0677
0.30,2.042,29.2,0.0675
1.00,2.083,33.3,0.0677
3.00,2.042,29.2,0.0679


In [7]:
best_C2 = min(logreg2_scores, key=lambda c: logreg2_scores[c]["brier"])
logreg2 = make_logreg(best_C2).fit(X_train2, y_train)
valid["LogReg2Prob"] = logreg2.predict_proba(X_valid2)[:, 1]
results[f"logreg v2 (C={best_C2})"] = evaluate(valid, "LogReg2Prob")

gb2 = HistGradientBoostingClassifier(
    learning_rate=lr, max_depth=depth, min_samples_leaf=leaf,
    max_iter=200, l2_regularization=1.0, random_state=42,
).fit(X_train2, y_train)
valid["GB2Prob"] = gb2.predict_proba(X_valid2)[:, 1]
results[f"gboost v2 {best_gb}"] = evaluate(valid, "GB2Prob")

pd.Series(logreg2[-1].coef_[0], index=logreg2[:-1].get_feature_names_out()).sort_values().round(2)

GapToPolePctRank                        -1.00
LogGrid                                 -0.47
GridPosition                            -0.38
DriverAvgFinish                         -0.31
DriverPodiumRateRank                    -0.26
TeamPodiumRateRank                      -0.26
TeamAvgFinish                           -0.19
TeamAvgFinishRank                       -0.16
PitLaneStart                            -0.11
DriverAvgGrid                           -0.11
NoQualiTime                             -0.06
DriverPodiumRate                        -0.05
missingindicator_DriverPodiumRateRank    0.01
missingindicator_DriverAvgFinishRank     0.01
missingindicator_DriverPodiumRate        0.01
missingindicator_DriverAvgFinish         0.01
missingindicator_DriverAvgGrid           0.01
missingindicator_TeamAvgFinishRank       0.04
missingindicator_TeamPodiumRate          0.04
missingindicator_TeamAvgFinish           0.04
missingindicator_TeamPodiumRateRank      0.04
RacesSoFar                        

## Comparison on 2024

In [8]:
pd.DataFrame(results).T

,hits_per_race,perfect_races_pct,brier
baseline (grid top 3),2.000,20.8,0.0699
logreg (C=3.0),2.042,29.2,0.0729
"gboost (0.03, 2, 20)",1.875,16.7,0.0743
logreg v2 (C=0.3),2.042,29.2,0.0675
"gboost v2 (0.03, 2, 20)",1.917,25.0,0.0708


In [9]:
# Per race: where do models beat or lose to the grid?
per_race = pd.DataFrame({
    "baseline": podium_hits(valid, "BaselineProb"),
    "logreg": podium_hits(valid, "LogRegProb"),
    "logreg_v2": podium_hits(valid, "LogReg2Prob"),
})
per_race["GrandPrix"] = valid.drop_duplicates(["Year", "Round"]).set_index(["Year", "Round"])["GrandPrix"]
per_race[per_race["logreg_v2"] != per_race["baseline"]]

baseline  logreg  logreg_v2              GrandPrix
Year Round                                                    
2024 3             2       1          1  Australian Grand Prix
     4             2       3          3    Japanese Grand Prix
     5             2       2          3     Chinese Grand Prix
     10            3       2          2     Spanish Grand Prix
     18            2       3          3   Singapore Grand Prix

## Caveat
The validation set is only 24 races, so one race changes hits per race by 0.04. Small gaps in the table above are noise. Notebook 05 retrains the chosen model on 2022–24 and scores 2025 once.